In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# --- ユーザー設定項目 ---

# 1. 対象となるCSVファイルのパス（ご自身のパスに合わせてください．）
input_file_path ='ss' 

# --- 設定ここまで ---

try:
    # ファイルを読み込む
    df = pd.read_csv(input_file_path)

    # --- 現在の列名を表示 ---
    print("--- 現在の列名 ---")
    print(df.columns.tolist())
    print("--------------------")

except FileNotFoundError:
    print(f"エラー: 入力ファイルが見つかりません。パスを確認してください: {input_file_path}")
except Exception as e:
    print(f"エラーが発生しました: {e}")

In [ ]:
!pip install japanize-matplotlib
import japanize_matplotlib

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/research/Opensubtitles2024en-ja/OpenSubtitles_Output_Manual/good/test_set.csv') # あなたのテストセットのファイル名
print(df.columns.tolist())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- ユーザー設定項目 ---

# 1. 評価に使用するテストセットのファイルパス
# ご自身のパスに合わせてください．
test_set_path = './test_sample.csv' 

# 2. 出力するグラフのファイル名
# ご自身のパスに合わせてください．
output_graph_path = './test_sample.png'

# --- 設定ここまで ---

# 日本語フォントの設定 (Google Colab/Jupyter環境で必要)
# もしローカル環境で日本語フォントがインストールされている場合は、そのフォント名に変更してください
try:
    import japanize_matplotlib
    print("japanize_matplotlibを適用しました。")
except ImportError:
    print("日本語表示のために 'pip install japanize-matplotlib' の実行を推奨します。")


try:
    print(f"テストセット ({test_set_path}) を読み込んでいます...")
    df = pd.read_csv(test_set_path)
    print("読み込み完了。")

    score_columns = [ 'bert_score_original_en_vs_basemodel_en', 'bleu_score_original_en_vs_basemodel_en', 'bert_score_original_ja_vs_basemodel_en']
    # --- 1. 定量的評価 (統計量の表) ---
    print("\n--- ベースラインモデル性能の定量的評価 (テストセット) ---")

    # .describe() を使って主要な統計量を計算
    # BLEUスコアが0のデータは、平均値を歪める可能性があるため、0を除いた平均も別途計算
    bleu_mean_nonzero = df[df['bleu_score_original_en_vs_basemodel_en'] > 0]['bleu_score_original_en_vs_basemodel_en'].mean()

    stats = df[score_columns].describe().loc[['mean', 'std', 'min', '50%', 'max']]
    stats = stats.rename(index={'50%': 'median'})

    # 計算したBLEUの平均値（0を除く）を表に追加
    stats.loc['mean (bleu > 0)', 'bleu_score_original_en_vs_basemodel_en'] = bleu_mean_nonzero

    # 表示フォーマットを整えて出力
    print(stats.to_string(float_format="%.4f"))
    print("----------------------------------------------------")


    # --- 2. 定性的評価 (グラフによる可視化) ---
    print("\nグラフを生成中...")
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle('Basemodel Grapth (testset)', fontsize=20, y=1.03)

    # a) BLEUスコアのヒストグラム
    sns.histplot(df['bleu_score_original_en_vs_basemodel_en'], bins=100, ax=axes[0, 0], color='skyblue', kde=False)
    axes[0, 0].set_title('BLEUScore Frequency (Original English_vs_Basemodel English)', fontsize=14)
    axes[0, 0].set_xlabel('BLEUScore', fontsize=12)
    axes[0, 0].set_ylabel('Frequency', fontsize=12)
    axes[0, 0].axvline(stats.loc['mean','bleu_score_original_en_vs_basemodel_en'], color='red', linestyle='--', label=f"ave: {stats.loc['mean','bleu_score_original_en_vs_basemodel_en']:.2f}")
    axes[0, 0].legend()

    # b) BERTScore (MT vs Ref) のヒストグラム
    sns.histplot(df['bert_score_original_en_vs_basemodel_en'], bins=50, ax=axes[0, 1], color='salmon', kde=True)
    axes[0, 1].set_title('BERTScore Frequency (Original English_vs_Basemodel English)', fontsize=14)
    axes[0, 1].set_xlabel('BERTScore (F1)', fontsize=12)
    axes[0, 1].set_ylabel('Frequuency', fontsize=12)
    axes[0, 1].axvline(stats.loc['mean', 'bert_score_original_en_vs_basemodel_en'], color='red', linestyle='--', label=f"ave: {stats.loc['mean','bert_score_original_en_vs_basemodel_en']:.4f}")
    axes[0, 1].legend()

    # c) BERTScore (Src vs Hyp) のヒストグラム
    sns.histplot(df['bert_score_original_ja_vs_basemodel_en'], bins=50, ax=axes[1, 0], color='lightgreen', kde=True)
    axes[1, 0].set_title('BERTScore Frequency (Original Japanese_vs_Basemodel English)', fontsize=14)
    axes[1, 0].set_xlabel('BERTScore (F1)', fontsize=12)
    axes[1, 0].set_ylabel('Frequency', fontsize=12)
    axes[1, 0].axvline(stats.loc['mean', 'bert_score_original_ja_vs_basemodel_en'], color='red', linestyle='--', label=f"ave: {stats.loc['mean','bert_score_original_ja_vs_basemodel_en']:.4f}")
    axes[1, 0].legend()

    # d) 3つのスコアの比較 (箱ひげ図)
    # スコアの範囲が大きく異なるため、正規化して比較
    df_normalized = df[score_columns].apply(lambda x: (x - x.min()) / (x.max() - x.min()))
    sns.boxplot(data=df_normalized, ax=axes[1, 1], palette='pastel')
    axes[1, 1].set_title('Comparison of distribution for each score (after normalization))', fontsize=14)
    axes[1, 1].set_ylabel('Normalized score (0 to 1)', fontsize=12)

    # グラフのレイアウトを調整
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    # グラフを画像ファイルとして保存
    plt.savefig(output_graph_path, dpi=300, bbox_inches='tight') # 高解像度で保存

    print(f"\n評価レポートのグラフが '{output_graph_path}' として保存されました。")
    print("\n--- 処理完了 ✨ ---")

except FileNotFoundError:
    print(f"エラー: 入力ファイルが見つかりません。パスを確認してください: {test_set_path}")
except Exception as e:
    print(f"エラーが発生しました: {e}")